# The `plot_proj` Idea for Discrete Distributions and True Measures

This notebook mirrors the QMCPy `plot_proj_function.ipynb` demo for projection
plots of discrete distributions and true measures.

Original QMCPy demo: [`QMCPy/demos/plot_proj_function.ipynb`](../../QMCPy/demos/plot_proj_function.ipynb)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QMCSoftware/QMC.jl/blob/develop/demos/plot_proj_function.ipynb)

*Note: The Python version uses `qp.plot_proj()` for matplotlib scatter plots.
This Julia version provides numerical summaries and lightweight text-based
visualizations of the same projection ideas so the demo remains executable in
headless environments.*

In [1]:
using QMC
import QMC: Uniform
using Statistics
using Printf

## The Following Examples Show Different Discrete Distribution Objects

Start with two-dimensional projections of several discrete distributions.

In [2]:
let n = 128
    println("=== 2D Point Set Statistics (n=$n) ===\n")
    for (name, dd) in [
        ("IID",         IIDStdUniform(2; seed=7)),
        ("Halton",      Halton(2; seed=7)),
        ("DigitalNetB2", DigitalNetB2(2; seed=7)),
        ("Lattice",     Lattice(2; seed=7)),
    ]
        pts = gen_samples(dd, n)
        println("$name:")
        for dim in 1:2
            col = pts[:, dim]
            println("  Dim $dim: mean=$(round(mean(col), digits=4)), " *
                    "std=$(round(std(col), digits=4)), " *
                    "min=$(round(minimum(col), digits=4)), " *
                    "max=$(round(maximum(col), digits=4))")
        end
        # L2 star discrepancy proxy: max gap between sorted points
        for dim in 1:2
            s = sort(pts[:, dim])
            gaps = diff(s)
            println("  Dim $dim max gap: $(round(maximum(gaps), digits=4)), " *
                    "mean gap: $(round(mean(gaps), digits=4))")
        end
        println("  Correlation(d1,d2): $(round(cor(pts[:,1], pts[:,2]), digits=4))")
        println()
    end
end

=== 2D Point Set Statistics (n=128) ===



IID:
  Dim 1: mean=0.4644, std=0.2706, min=0.0, max=0.9875


  Dim 2: mean=0.4993, std=0.2764, min=0.0031, max=0.9869
  Dim 1 max gap: 0.0587, mean gap: 0.0078


  Dim 2 max gap: 0.0389, mean gap: 0.0077
  Correlation(d1,d2): 0.0027

Halton:


  Dim 1: mean=0.4967, std=0.2898, min=0.0006, max=0.9928
  Dim 2: mean=0.4992, std=0.2887, min=0.0019, max=0.9895
  Dim 1 max gap: 0.0078, mean gap: 0.0078
  Dim 2 max gap: 0.0123, mean gap: 0.0078
  Correlation(d1,d2): -0.0096

DigitalNetB2:


  Dim 1: mean=0.5006, std=0.2898, min=0.0045, max=0.9967
  Dim 2: mean=0.4992, std=0.2898, min=0.0031, max=0.9953
  Dim 1 max gap: 0.0078, mean gap: 0.0078
  Dim 2 max gap: 0.0078, mean gap: 0.0078
  Correlation(d1,d2): -0.0002

Lattice:


  Dim 1: mean=0.5029, std=0.2898, min=0.0068, max=0.999
  Dim 2: mean=0.5027, std=0.2898, min=0.0066, max=0.9988
  Dim 1 max gap: 0.0078, mean gap: 0.0078
  Dim 2 max gap: 0.0078, mean gap: 0.0078
  Correlation(d1,d2): -0.0287



## Higher-Dimensional Projections

For 4-dimensional sequences, inspect all pairwise 2-dimensional projections.

In [3]:
let n = 128, d = 4
    for (name, dd) in [
        ("Halton",       Halton(d; seed=7)),
        ("DigitalNetB2", DigitalNetB2(d; seed=7)),
        ("Lattice",      Lattice(d; seed=7)),
    ]
        pts = gen_samples(dd, n)
        println("=== $name ($d D, $n points) ===")
        println("Pairwise correlations:")
        for i in 1:d
            row = [@sprintf("% .4f", cor(pts[:,i], pts[:,j])) for j in 1:d]
            println("  ", join(row, "  "))
        end
        println()
        println("Per-dimension statistics:")
        for dim in 1:d
            col = pts[:, dim]
            s = sort(col)
            max_gap = maximum(diff(s))
            @printf("  Dim %d: mean=%.4f, max_gap=%.4f\n", dim, mean(col), max_gap)
        end
        println()
    end
end

=== Halton (4 D, 128 points) ===


Pairwise correlations:
   1.0000  -0.0096  -0.0170   0.0073
  -0.0096   1.0000  -0.0168   0.0125
  -0.0170  -0.0168   1.0000   0.0596
   0.0073   0.0125   0.0596   1.0000

Per-dimension statistics:
  Dim 1: mean=0.4967, max_gap=0.0078


  Dim 2: mean=0.4992, max_gap=0.0123
  Dim 3: mean=0.4998, max_gap=0.0080
  Dim 4: mean=0.5008, max_gap=0.0117

=== DigitalNetB2 (4 D, 128 points) ===
Pairwise correlations:
   1.0000  -0.0002   0.0005   0.0035
  -0.0002   1.0000   0.0002   0.0002
   0.0005   0.0002   1.0000  -0.0108
   0.0035   0.0002  -0.0108   1.0000

Per-dimension statistics:
  Dim 1: mean=0.5002, max_gap=0.0078
  Dim 2: mean=0.5001, max_gap=0.0078
  Dim 3: mean=0.4980, max_gap=0.0078
  Dim 4: mean=0.5001, max_gap=0.0078

=== Lattice (4 D, 128 points) ===
Pairwise correlations:
   1.0000  -0.0687  -0.0071   0.0255
  -0.0687   1.0000  -0.0255   0.0657
  -0.0071  -0.0255   1.0000  -0.0778
   0.0255   0.0657  -0.0778   1.0000

Per-dimension statistics:
  Dim 1: mean=0.5028, max_gap=0.0078
  Dim 2: mean=0.5030, max_gap=0.0078
  Dim 3: mean=0.5018, max_gap=0.0078
  Dim 4: mean=0.5023, max_gap=0.0078



## The Following Examples Show Different True Measure Objects

Apply true measures such as Gaussian transforms to the underlying point sets.

In [4]:
# Gaussian transform of IID points
dd = IIDStdUniform(2; seed=7)
tm = Gaussian(dd; mean=[2.0, 4.0], covariance=[9.0 4.0; 4.0 5.0])
n = 256
x_uniform = gen_samples(dd, n)
x_transformed = zeros(n, 2)
for i in 1:n
    x_transformed[i, :] = vec(transform(tm, x_uniform[i:i, :]))
end

println("Gaussian(mean=[2,4], cov=[[9,4],[4,5]]) transform:")
println("  Dim 1: mean=$(round(mean(x_transformed[:,1]), digits=2)), " *
        "std=$(round(std(x_transformed[:,1]), digits=2)) (target: mean=2, std=3)")
println("  Dim 2: mean=$(round(mean(x_transformed[:,2]), digits=2)), " *
        "std=$(round(std(x_transformed[:,2]), digits=2)) (target: mean=4, std≈2.24)")
println("  Correlation: $(round(cor(x_transformed[:,1], x_transformed[:,2]), digits=3)) " *
        "(target: $(round(4/sqrt(9*5), digits=3)))")

Gaussian(mean=[2,4], cov=[[9,4],[4,5]]) transform:


  Dim 1: mean=2.2, std=2.84 (target: mean=2, std=3)
  Dim 2: mean=4.06, std=2.2 (target: mean=4, std≈2.24)
  Correlation: 0.593 (target: 0.596)


## Text-Based Projection View

A simple ASCII projection view to visualize the same structure without a full plotting backend.

In [5]:
function ascii_scatter(x, y; width=60, height=20, title="")
    xmin, xmax = minimum(x), maximum(x)
    ymin, ymax = minimum(y), maximum(y)
    xr = xmax - xmin; yr = ymax - ymin
    if xr == 0; xr = 1.0; end
    if yr == 0; yr = 1.0; end
    grid = fill(' ', height, width)
    for i in eachindex(x)
        c = clamp(round(Int, (x[i] - xmin) / xr * (width - 1)) + 1, 1, width)
        r = clamp(height - round(Int, (y[i] - ymin) / yr * (height - 1)), 1, height)
        grid[r, c] = '·'
    end
    !isempty(title) && println(title)
    for r in 1:height
        println(String(grid[r, :]))
    end
    @printf("x: [%.2f, %.2f]  y: [%.2f, %.2f]\n", xmin, xmax, ymin, ymax)
end

# Compare IID vs Lattice
n = 256
dd_iid = IIDStdUniform(2; seed=42)
dd_lat = Lattice(2; seed=42)
x_iid = gen_samples(dd_iid, n)
x_lat = gen_samples(dd_lat, n)

ascii_scatter(x_iid[:,1], x_iid[:,2]; title="\nIID ($n points):")
println()
ascii_scatter(x_lat[:,1], x_lat[:,2]; title="Lattice ($n points):")


IID (256 points):
  ·           ··          ·        ·      ·  · ·      ·     
     ·     ·      ··  ·     ··                    ·        ·
      ·            ·      · ·         ·          ···        
·               · ·     ··  ·  ·        · ··   ·  · · ···   
         ·   ··    ·   ·   · ·   ·         ·      ·· ·      
  ·  ·    · ·                   ··    · ·   ···    ·  ·     
        ··  ·    ·        ·      ·  ·   · ·   · ·  · · ·   ·
      ·   ·  ·    · ···   ··  ·        ··   ·· ·  ·       · 
  ··       ·               ·    ·      ·  ·     ·    ·      
       ·   ·    ·   ·    ·  ·    ·    ·   ·           ·     
   ·· ·  ··  ·    · · ·        ·  ··  ·  ·      ··    ·     
   ·   ·     ·         ·           ·  ·     ·  ··          ·
         ·          ·         ·· · ·      ·     ·        ·  
      ···  ·     ·       ·         ·   ·      ·           · 
 · ·    · ·· ·      ·     ·        · ·· ·              ·  · 
  ·         ·     ·    ·         ·      · ·      · ·  ·     
    ·